# Milestone 1: EDA + Random Baseline

Exploratory Data Analysis of the genre classification dataset and random baseline submission.

In [ ]:
!pip install transformers wandb librosa xgboost -q

In [ ]:
import sys, os, random
sys.path.append('../src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import librosa, librosa.display, torchaudio
import IPython.display as ipd
from utils import *

set_seed()
paths = get_paths()
print(f"BASE_DIR:    {paths['BASE_DIR']}")
print(f"STEMS_DIR:   {paths['STEMS_DIR']}")
print(f"MASHUPS_DIR: {paths['MASHUPS_DIR']}")
print(f"NOISE_DIR:   {paths['NOISE_DIR']}")

## Directory Exploration

In [ ]:
print("=" * 50)
print("TRAINING DATA (Stems per Genre)")
print("=" * 50)
for genre in GENRES:
    genre_path = os.path.join(paths['STEMS_DIR'], genre)
    if os.path.exists(genre_path):
        songs = sorted(os.listdir(genre_path))
        first_stems = os.listdir(os.path.join(genre_path, songs[0])) if songs else []
        print(f"  {genre:12s}: {len(songs)} songs | stems: {first_stems}")

print()
print("=" * 50)
print("TEST DATA (Mashup Files)")
print("=" * 50)
mashups = sorted(os.listdir(paths['MASHUPS_DIR']))
print(f"Total mashup files: {len(mashups)}")
print(f"First 10: {mashups[:10]}")
print(f"Last 5:   {mashups[-5:]}")

print()
print("=" * 50)
print("test.csv and sample_submission.csv")
print("=" * 50)
test_df = pd.read_csv(paths['TEST_CSV'])
print(f"test.csv shape: {test_df.shape}")
print(f"Columns: {list(test_df.columns)}")
print(test_df.head())
print(f"\nID dtype: {test_df['id'].dtype}")

if os.path.exists(paths['SAMPLE_SUB']):
    sample_sub = pd.read_csv(paths['SAMPLE_SUB'])
    print(f"\nsample_submission.csv:\n{sample_sub.head()}")

if paths['NOISE_DIR']:
    noise_files = [f for f in os.listdir(paths['NOISE_DIR']) if f.endswith('.wav')]
    print(f"\nNoise files (ESC-50): {len(noise_files)}")

## EDA — Class Distribution + Audio Stats

In [ ]:
# Count songs per genre
genre_counts = {}
for genre in GENRES:
    genre_path = os.path.join(paths['STEMS_DIR'], genre)
    if os.path.exists(genre_path):
        genre_counts[genre] = len(os.listdir(genre_path))

plt.figure(figsize=(10, 5))
plt.bar(genre_counts.keys(), genre_counts.values(), color='steelblue')
plt.title('Number of Songs per Genre (Training Data)')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Total training songs: {sum(genre_counts.values())}")
print(f"Songs per genre: {genre_counts}")

# Audio stats
print("\nAudio Stats (3 random samples):")
for genre in random.sample(GENRES, 3):
    genre_path = os.path.join(paths['STEMS_DIR'], genre)
    song = random.choice(os.listdir(genre_path))
    vocal_path = os.path.join(genre_path, song, 'vocals.wav')
    if os.path.exists(vocal_path):
        info = torchaudio.info(vocal_path)
        dur = info.num_frames / info.sample_rate
        print(f"  {genre}/{song}: SR={info.sample_rate}, Duration={dur:.1f}s, Channels={info.num_channels}")

# Check for missing stems
print("\nChecking for missing stems...")
missing_count = 0
stem_names = ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']
for genre in GENRES:
    genre_path = os.path.join(paths['STEMS_DIR'], genre)
    if os.path.exists(genre_path):
        for song in os.listdir(genre_path):
            for stem in stem_names:
                if not os.path.exists(os.path.join(genre_path, song, stem)):
                    missing_count += 1
                    if missing_count <= 5:
                        print(f"  MISSING: {genre}/{song}/{stem}")
print(f"Total missing stems: {missing_count}")

## EDA — Visualize Audio (Clean Stem vs Noisy Mashup)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Clean stem waveform + spectrogram
sample_genre = 'rock'
song_folders = sorted(os.listdir(os.path.join(paths['STEMS_DIR'], sample_genre)))
sample_song = song_folders[0]
clean_path = os.path.join(paths['STEMS_DIR'], sample_genre, sample_song, 'vocals.wav')
y_clean, sr_clean = librosa.load(clean_path, sr=SAMPLE_RATE, duration=5.0)

axes[0, 0].set_title(f'Clean Stem: {sample_genre}/{sample_song} (vocals)')
librosa.display.waveshow(y_clean, sr=sr_clean, ax=axes[0, 0], color='blue')
axes[0, 0].set_xlabel('Time (s)'); axes[0, 0].set_ylabel('Amplitude')

S_clean = librosa.feature.melspectrogram(y=y_clean, sr=sr_clean, n_mels=64)
S_clean_db = librosa.power_to_db(S_clean, ref=np.max)
img1 = librosa.display.specshow(S_clean_db, sr=sr_clean, x_axis='time', y_axis='mel', ax=axes[0, 1])
axes[0, 1].set_title('Mel Spectrogram (Clean Stem)')
fig.colorbar(img1, ax=axes[0, 1], format='%+2.0f dB')

# Mashup waveform + spectrogram
mashup_files_list = sorted(os.listdir(paths['MASHUPS_DIR']))
sample_mashup = mashup_files_list[0]
mashup_path = os.path.join(paths['MASHUPS_DIR'], sample_mashup)
y_mashup, sr_mashup = librosa.load(mashup_path, sr=SAMPLE_RATE, duration=5.0)

axes[1, 0].set_title(f'Test Mashup: {sample_mashup}')
librosa.display.waveshow(y_mashup, sr=sr_mashup, ax=axes[1, 0], color='red')
axes[1, 0].set_xlabel('Time (s)'); axes[1, 0].set_ylabel('Amplitude')

S_mashup = librosa.feature.melspectrogram(y=y_mashup, sr=sr_mashup, n_mels=64)
S_mashup_db = librosa.power_to_db(S_mashup, ref=np.max)
img2 = librosa.display.specshow(S_mashup_db, sr=sr_mashup, x_axis='time', y_axis='mel', ax=axes[1, 1])
axes[1, 1].set_title('Mel Spectrogram (Noisy Mashup)')
fig.colorbar(img2, ax=axes[1, 1], format='%+2.0f dB')

plt.tight_layout()
plt.show()

print("Clean stem audio:")
ipd.display(ipd.Audio(clean_path))
print("\nTest mashup audio:")
ipd.display(ipd.Audio(mashup_path))

## Random Baseline Submission

In [ ]:
submission_df = pd.read_csv(paths['SAMPLE_SUB'])
submission_df['genre'] = [random.choice(GENRES) for _ in range(len(submission_df))]
submission_df.to_csv('submission_random.csv', index=False)
print("Random baseline submission created: submission_random.csv")
print(f"Shape: {submission_df.shape}")
print(submission_df.head(10))
print("\nSubmit this to Kaggle to get your first score (~0.10 expected).")